# Phase B.2.2 — Directed probes on Gemma 2 9B (layers 20 and 31)

**Prerequisite:** B.2.1 (`exp_gemma9b_lsi.ipynb`) PASSED — `gemma9b_lsi_layer20_31.json` in the directory.

**Goal:** Identify discriminative features for 3 morphosyntactic phenomena on 9B at L20 and L31, and classify the locality regime in each (phenomenon × layer). Mirrors the methodology of `exp_probes_multilayer.ipynb` (on 2B).

**Setup:**
- Phenomena: **gender, crase, clitics** (3 — selected as the most informative subset from 2B: gender = semi-localized, crase = semi/distributed, clitics = localized).
- Pairs per phenomenon: 10 positive + 10 negative (identical to 2B for comparability).
- Layers: 20 (~48%) and 31 (~74%).
- Metric: top-K by |diff| of mean positive − negative activations; locality regime classified by $\text{conc}_{1/5} = \text{top1} / \text{top5}$.

**Outputs:**
- `gemma9b_probes_layer20_31.json` — top-20 features and regime per (phenomenon × layer).

**Next:** B.2.3 (`exp_gemma9b_steering.ipynb`) uses the candidate features from this JSON for the gender causal benchmark (228 items).

## 1. Instalação e ambiente

In [ ]:
!pip install -q sae-lens transformer-lens

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    name = torch.cuda.get_device_name(0)
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU:    {name} ({total_gb:.1f} GB total)")
    assert total_gb >= 20, "Precisa de >=20GB VRAM em bf16. Use L4 ou A100."
else:
    raise RuntimeError("Precisa de GPU.")

## 2. Configuração

In [ ]:
MODEL_NAME = "gemma-2-9b"
HF_MODEL_ID = f"google/{MODEL_NAME}"
LAYERS = [20, 31]
WIDTH = "16k"
SAE_RELEASE = "gemma-scope-9b-pt-res-canonical"

DTYPE = torch.bfloat16
TOP_K = 20

REGIME_THRESHOLDS = {"localized": 0.45, "semi_localized": 0.25}

OUT_JSON = "gemma9b_probes_layer20_31.json"

print(f"Model:   {MODEL_NAME}")
print(f"Layers:  {LAYERS}")
print(f"Top-K:   {TOP_K}")
print(f"Regime:  localized>={REGIME_THRESHOLDS['localized']}, semi>={REGIME_THRESHOLDS['semi_localized']}")

def classify_regime(conc):
    if conc >= REGIME_THRESHOLDS["localized"]:
        return "localized"
    elif conc >= REGIME_THRESHOLDS["semi_localized"]:
        return "semi-localized"
    return "distributed"

## 3. Load model and SAEs

In [ ]:
import gc
from transformers import AutoModelForCausalLM, AutoTokenizer

torch.cuda.empty_cache()
gc.collect()

print(f"Carregando {HF_MODEL_ID} em {DTYPE} (~4-6 min na L4)...")
tokenizer = AutoTokenizer.from_pretrained(HF_MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    HF_MODEL_ID,
    torch_dtype=DTYPE,
    device_map={"": device},
    low_cpu_mem_usage=True,
    attn_implementation="eager",
)
model.eval()
for p in model.parameters():
    p.requires_grad_(False)

n_model_layers = model.config.num_hidden_layers
d_model = model.config.hidden_size
print(f"layers: {n_model_layers} | d_model: {d_model}")
print(f"VRAM em uso: {torch.cuda.memory_allocated()/1024**3:.2f} GB")

In [ ]:
from sae_lens import SAE

saes = {}
for L in LAYERS:
    sae_id = f"layer_{L}/width_{WIDTH}/canonical"
    print(f"Carregando SAE {sae_id}...")
    s = SAE.from_pretrained(release=SAE_RELEASE, sae_id=sae_id, device=device)
    if isinstance(s, tuple):
        s = s[0]
    s = s.to(DTYPE)
    saes[L] = s
    assert s.cfg.d_in == d_model

D_SAE = saes[LAYERS[0]].cfg.d_sae
print(f"VRAM em uso: {torch.cuda.memory_allocated()/1024**3:.2f} GB | d_sae={D_SAE}")

## 4. Hooks multi-layer

In [ ]:
_cap = {}
_handles = []

def _make_hook(layer_idx):
    def hook(module, inputs, output):
        h = output[0] if isinstance(output, tuple) else output
        _cap[layer_idx] = h
    return hook

def install_hooks(layers):
    for h in _handles:
        h.remove()
    _handles.clear()
    _cap.clear()
    for L in layers:
        handle = model.model.layers[L].register_forward_hook(_make_hook(L))
        _handles.append(handle)

def encode_text_multilayer(text, layers):
    """Retorna {layer: (T, d_sae)} para um único texto."""
    _cap.clear()
    enc = tokenizer(text, return_tensors="pt").to(device)
    with torch.no_grad():
        model(**enc, use_cache=False)
    out = {}
    for L in layers:
        resid = _cap[L]
        out[L] = saes[L].encode(resid)[0]
    return out

install_hooks(LAYERS)
_test = encode_text_multilayer("O Brasil é o maior país da América do Sul.", LAYERS)
for L in LAYERS:
    l0 = (_test[L] > 0).float().sum(dim=-1).mean().item()
    print(f"Layer {L}: shape={tuple(_test[L].shape)} | L0={l0:.1f}")

## 5. Definição dos probes (idêntica ao 2B)

In [ ]:
PROBES = {
    "gender": {
        "positive": [
            "A menina bonita correu pelo parque.",
            "A professora dedicada ensinou a lição.",
            "A diretora nova chegou na escola.",
            "A médica experiente atendeu o paciente.",
            "A engenheira brasileira projetou a ponte.",
            "A menina inteligente resolveu o problema.",
            "A advogada famosa venceu o caso.",
            "A cantora talentosa encantou a plateia.",
            "A cozinheira habilidosa preparou o jantar.",
            "A estudante aplicada passou no exame.",
        ],
        "negative": [
            "O menino bonito correu pelo parque.",
            "O professor dedicado ensinou a lição.",
            "O diretor novo chegou na escola.",
            "O médico experiente atendeu o paciente.",
            "O engenheiro brasileiro projetou a ponte.",
            "O menino inteligente resolveu o problema.",
            "O advogado famoso venceu o caso.",
            "O cantor talentoso encantou a plateia.",
            "O cozinheiro habilidoso preparou o jantar.",
            "O estudante aplicado passou no exame.",
        ],
    },
    "crase": {
        "positive": [
            "Ele foi à escola todos os dias.",
            "Ela se referiu à diretora do colégio.",
            "O aluno entregou o trabalho à professora.",
            "Nós fomos à praia no fim de semana.",
            "Ele assistiu à palestra com atenção.",
            "Ela voltou à cidade natal depois de anos.",
            "O diretor respondeu à carta do ministro.",
            "Nós nos dirigimos à saída de emergência.",
            "Ela se dedicou à pesquisa por décadas.",
            "O turista foi à catedral mais antiga.",
        ],
        "negative": [
            "Ele foi ao mercado todos os dias.",
            "Ela se referiu ao diretor do colégio.",
            "O aluno entregou o trabalho ao professor.",
            "Nós fomos ao parque no fim de semana.",
            "Ele assistiu ao jogo com atenção.",
            "Ele voltou ao escritório depois de anos.",
            "O diretor respondeu ao ofício do ministro.",
            "Nós nos dirigimos ao portão de emergência.",
            "Ele se dedicou ao estudo por décadas.",
            "O turista foi ao museu mais antigo.",
        ],
    },
    "clitics": {
        "positive": [
            "Ele me disse a verdade ontem.",
            "O professor nos explicou a matéria.",
            "Ela se arrumou para a festa.",
            "Ele lhe entregou o presente.",
            "Eles nos convidaram para o jantar.",
            "A mãe me chamou para almoçar.",
            "O chefe nos pediu para ficar.",
            "Ela se preparou cuidadosamente.",
            "Ele lhe prometeu que voltaria.",
            "Eles nos avisaram do perigo.",
        ],
        "negative": [
            "He told me the truth yesterday.",
            "The teacher explained the subject to us.",
            "She got ready for the party.",
            "He gave her the present.",
            "They invited us for dinner.",
            "Mom called me for lunch.",
            "The boss asked us to stay.",
            "She prepared herself carefully.",
            "He promised her he would return.",
            "They warned us of the danger.",
        ],
    },
}

for p, v in PROBES.items():
    print(f"  {p}: {len(v['positive'])} positivos x {len(v['negative'])} negativos")

## 6. Computar ativações médias por texto e features discriminativas

In [ ]:
from tqdm.auto import tqdm

def mean_acts_per_text(text, layers):
    feats = encode_text_multilayer(text, layers)
    return {L: feats[L].float().mean(dim=0).cpu() for L in layers}

def find_discriminative(pos_texts, neg_texts, layers, top_k=TOP_K):
    pos = {L: [] for L in layers}
    neg = {L: [] for L in layers}
    for t in tqdm(pos_texts, desc="  pos", leave=False):
        for L, v in mean_acts_per_text(t, layers).items():
            pos[L].append(v)
    for t in tqdm(neg_texts, desc="  neg", leave=False):
        for L, v in mean_acts_per_text(t, layers).items():
            neg[L].append(v)
    out = {}
    for L in layers:
        P = torch.stack(pos[L])
        N = torch.stack(neg[L])
        mean_pos = P.mean(dim=0)
        mean_neg = N.mean(dim=0)
        std_pos = P.std(dim=0)
        std_neg = N.std(dim=0)
        diff = mean_pos - mean_neg
        pooled_std = torch.sqrt((std_pos**2 + std_neg**2) / 2 + 1e-8)
        effect_size = diff / pooled_std
        top_idx = diff.abs().argsort(descending=True)[:top_k]
        out[L] = {
            "top_features": [
                {
                    "feature_id": int(idx.item()),
                    "diff": float(diff[idx].item()),
                    "effect_size": float(effect_size[idx].item()),
                    "mean_pos": float(mean_pos[idx].item()),
                    "mean_neg": float(mean_neg[idx].item()),
                }
                for idx in top_idx
            ],
            "max_abs_diff": float(diff.abs().max().item()),
            "max_effect_size": float(effect_size[top_idx[0]].item()),
        }
    return out

PROBE_RESULTS = {}
for phenomenon, texts in PROBES.items():
    print(f"\n=== {phenomenon} ===")
    PROBE_RESULTS[phenomenon] = find_discriminative(texts["positive"], texts["negative"], LAYERS)
    for L in LAYERS:
        top1 = PROBE_RESULTS[phenomenon][L]["top_features"][0]
        print(f"  L{L} top-1: feat {top1['feature_id']:>5} | diff {top1['diff']:+.3f} | d {top1['effect_size']:+.3f}")

## 7. Locality analysis per (phenomenon × layer)

In [ ]:
LOCALITY = {}
print(f"{'fenômeno':<14}{'layer':>7}{'top1':>10}{'top5':>10}{'top20':>10}{'conc_1/5':>10}  regime")
print("-" * 78)
for phenomenon in PROBES.keys():
    LOCALITY[phenomenon] = {}
    for L in LAYERS:
        diffs = [abs(f["diff"]) for f in PROBE_RESULTS[phenomenon][L]["top_features"][:20]]
        top1 = diffs[0]
        top5 = sum(diffs[:5])
        top20 = sum(diffs[:20])
        conc = top1 / (top5 + 1e-8)
        regime = classify_regime(conc)
        LOCALITY[phenomenon][L] = {
            "top1": top1, "top5": top5, "top20": top20,
            "conc_1_5": conc, "regime": regime,
        }
        print(f"{phenomenon:<14}{L:>7}{top1:>10.3f}{top5:>10.3f}{top20:>10.3f}{conc:>10.3f}  {regime}")

## 8. Identifying "generic" (high-norm) features

On 2B we observed that features such as 12102/11248/7127 appeared as top-1 in 4-5 different phenomena — they were high-norm features, not phenomenon-specific. Here we detect the same pattern on 9B (if it exists) so that B.2.3 (steering) can use a fallback.

In [ ]:
from collections import Counter

GENERIC = {}
for L in LAYERS:
    cnt = Counter()
    for phenomenon in PROBES.keys():
        fid = PROBE_RESULTS[phenomenon][L]["top_features"][0]["feature_id"]
        cnt[fid] += 1
    generic_ids = [fid for fid, c in cnt.items() if c >= 2]
    GENERIC[L] = generic_ids
    print(f"Layer {L}:")
    for fid, c in cnt.most_common():
        if c >= 2:
            phs = [p for p in PROBES if PROBE_RESULTS[p][L]['top_features'][0]['feature_id'] == fid]
            print(f"  feat {fid:>5} aparece como top-1 em {c} fenômenos: {phs}")
    if not generic_ids:
        print("  (nenhuma feature \"genérica\" detectada)")

## 9. Comparação 2B (referência) vs 9B

Regimes do 2B vêm de `exp_multilayer_focused_results.json` (camadas 9, 13, 17).

In [ ]:
REFS_2B = {
    "gender":  {"9": "semi-localized", "13": "semi-localized", "17": "semi-localized"},
    "crase":   {"9": "semi-localized", "13": "semi-localized", "17": "distributed"},
    "clitics": {"9": "localized",      "13": "localized",      "17": "semi-localized"},
}

print("== Regimes 2B (referência: paper, camadas 9, 13, 17) ==")
print(f"  {'fenômeno':<10}{'L9':>16}{'L13':>16}{'L17':>16}")
for ph, regs in REFS_2B.items():
    print(f"  {ph:<10}{regs['9']:>16}{regs['13']:>16}{regs['17']:>16}")

print("\n== Regimes 9B (este experimento) ==")
print(f"  {'fenômeno':<10}{'L20':>16}{'L31':>16}")
for ph in PROBES.keys():
    r20 = LOCALITY[ph][20]["regime"]
    r31 = LOCALITY[ph][31]["regime"]
    print(f"  {ph:<10}{r20:>16}{r31:>16}")

## 10. Save results

In [ ]:
import json

result = {
    "experiment": "gemma9b_probes_multilayer",
    "model": MODEL_NAME,
    "sae_release": SAE_RELEASE,
    "layers": LAYERS,
    "n_model_layers": n_model_layers,
    "d_sae": D_SAE,
    "phenomena": list(PROBES.keys()),
    "regime_thresholds": REGIME_THRESHOLDS,
    "probes": PROBES,
    "probe_results": {
        ph: {str(L): PROBE_RESULTS[ph][L] for L in LAYERS} for ph in PROBES.keys()
    },
    "locality_analysis": {
        ph: {str(L): LOCALITY[ph][L] for L in LAYERS} for ph in PROBES.keys()
    },
    "generic_features_by_layer": {str(L): GENERIC[L] for L in LAYERS},
    "refs_2b": REFS_2B,
}

with open(OUT_JSON, "w") as f:
    json.dump(result, f, indent=2, ensure_ascii=False)

print(f"Salvo em {OUT_JSON}")
print()
print("=== regime summary ===")
for ph in PROBES.keys():
    for L in LAYERS:
        info = LOCALITY[ph][L]
        print(f"  {ph:<10} L{L:<3}: conc_1/5={info['conc_1_5']:.3f}  -> {info['regime']}")

## 11. Next steps

**Criterion to proceed to B.2.3:** have top-1/top-2 features with a reasonable |diff| (≥ 1.0) for gender, in at least one layer — without this, ablation on the 228-item benchmark has little signal.

**If the top-1 features are also "generic" (high-norm) on 9B:** B.2.3 will use the top-2 or top-3 (fallback) as the effective feature, as we did on 2B. Cell 8 of this notebook already flags which features are suspicious.

**For B.2.3:** this JSON provides, per (phenomenon × layer), the top-20 features with `feature_id`, `diff`, `effect_size`, `mean_pos`, `mean_neg`. The B.2.3 notebook will use these lists to choose the ablation feature per layer.